In [ ]:
import numpy as np
import pandas as pd

def scale_to_unit_box(x, y):
    x_min, x_max = np.min(x), np.max(x)
    y_min, y_max = np.min(y), np.max(y)

    x_range = x_max - x_min
    y_range = y_max - y_min

    if x_range >= y_range:
        scale = 2.0 / x_range  # target: [-1, 1]
        x_scaled = (x - x_min) * scale - 1.0
        y_scaled = (y - y_min) * scale - (y_range / x_range)
    else:
        scale = 2.0 / y_range
        y_scaled = (y - y_min) * scale - 1.0
        x_scaled = (x - x_min) * scale - (x_range / y_range)

    return x_scaled, y_scaled, scale

# ---------- Sampling Function ----------
def sample_points_in_circle(n, r_min=0.0, r_max=1.0, seed=42):
    rng = np.random.default_rng(seed)
    theta = rng.uniform(0, 2 * np.pi, n)
    r = np.sqrt(rng.uniform(r_min**2, r_max**2, n))  # uniform area
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    return np.stack([x, y], axis=1)

# ---------- Vector Fields ----------
dataset = {}
n = 1000
np.random.seed(42)

# --- 1. Rotation-only field ---
pos_rot = sample_points_in_circle(n, r_min=0.3, r_max=1.0)
x, y = pos_rot[:, 0], pos_rot[:, 1]
velocity_rot = np.stack([-y, x], axis=1)
theta = np.arctan2(y, x)
time_rot = ((theta + 2 * np.pi) % (2 * np.pi)) / (2 * np.pi)

dataset["rotation"] = {
    "position": pos_rot,
    "velocity": velocity_rot,
    "time": time_rot,
    "name": "rotation"
}

# --- 2. Spiral field ---
pos_spiral = sample_points_in_circle(n, r_min=0.0, r_max=1.0)
x, y = pos_spiral[:, 0], pos_spiral[:, 1]
velocity_spiral = np.stack([x - y, x + y], axis=1)
time_spiral = np.sqrt(x**2 + y**2)
time_spiral /= time_spiral.max()

dataset["spiral"] = {
    "position": pos_spiral,
    "velocity": velocity_spiral,
    "time": time_spiral,
    "name": "spiral"
}

In [ ]:
# --- 3. Hyperbolic field ---
# Define the saddle vector field
np.random.seed(42)
n_points = 1000
xlim = ylim = [-2, 2]

X = np.random.uniform(xlim[0], xlim[1], size=(n_points, 2))

# Construct saddle matrix A with eigenvalues -1, +1 along 45°
theta = np.pi / 4
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
Lambda = np.diag([-1, 1])
A = R @ Lambda @ R.T
# Compute vector field
V = X @ A.T

x, y = X[:, 0], X[:, 1]
# Normalize
x, y, scale = scale_to_unit_box(x, y)
X = np.column_stack([x, y])
V = V * scale  # Apply same scale to preserve direction


# Define time
unstable_dir = np.array([1, -1]) / np.sqrt(2)

# Project positions onto unstable direction
proj = X @ unstable_dir

# Use absolute value — origin is time = 0
time_saddle = np.abs(proj)
time_saddle /= time_saddle.max()  # normalize to [0, 1]

# Save to dataset
dataset["saddle"] = {
    "position": X,
    "velocity": V,
    "time": time_saddle,
    "name": "saddle"
}

In [ ]:
# --- 4. Quadratic source-sink field ---
def sample_points_in_ball(n, radius=3.0, seed=999):
    rng = np.random.default_rng(seed)
    theta = rng.uniform(0, 2 * np.pi, n)
    r = radius * np.sqrt(rng.uniform(0, 1, n))
    x = r * np.cos(theta)
    y = r * np.sin(theta)
    return np.stack([x, y], axis=1)

def quadratic_field(x, y):
    Bx = x**2 - y**2 - 4
    By = 2 * x * y
    return Bx, By

# Sample points
n = 1000
pos_q = sample_points_in_ball(n, radius=3.0)
x, y = pos_q[:, 0], pos_q[:, 1]

# Compute field
Bx, By = quadratic_field(x, y)
velocity_q = np.stack([Bx, By], axis=1)

# Define time: projection onto leftward axis
reference_dir = np.array([-1.0, 0.0])
proj = pos_q @ reference_dir
time_q = (proj - proj.min()) / (proj.max() - proj.min())


x, y, scale = scale_to_unit_box(x, y)
pos_q_scaled = np.column_stack([x, y])
velocity_q_scaled = velocity_q * scale  # scale velocity accordingly

# Recompute time using normalized positions
proj_scaled = pos_q_scaled @ reference_dir
time_q = (proj_scaled - proj_scaled.min()) / (proj_scaled.max() - proj_scaled.min())

# Save to dataset
dataset["quadratic_source_sink"] = {
    "position": pos_q_scaled,
    "velocity": velocity_q_scaled,
    "time": time_q,
    "name": "quadratic_source_sink"
}

In [ ]:
import matplotlib.pyplot as plt

# ---------- Visualize all 4 vector fields ----------
fig, axs = plt.subplots(2, 2, figsize=(12, 10))

for ax, (key, field) in zip(axs.ravel(), dataset.items()):
    pos = field["position"]
    vel = field["velocity"]
    time = field["time"]

    x, y = pos[:, 0], pos[:, 1]
    dx, dy = vel[:, 0], vel[:, 1]

    # Scatter points colored by time
    sc = ax.scatter(x, y, c=time, cmap='twilight', s=12, alpha=0.9, edgecolors='none')

    # Overlay vector field
    ax.quiver(x, y, dx, dy, angles='xy', scale_units='xy',
              scale=9, color='black', alpha=0.5, width=0.0025)

    ax.set_title(field["name"], fontsize=14)
    ax.set_aspect('equal')
    ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle("Vector Fields with Periodic Time Coloring", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import os

def save_all_datasets(dataset_dict, output_dir="./data/2d"):
    os.makedirs(output_dir, exist_ok=True)

    for key, data in dataset_dict.items():
        pos = data["position"]
        vel = data["velocity"]
        t   = data["time"]
        name = data["name"]

        df = pd.DataFrame({
            "x": pos[:, 0],
            "y": pos[:, 1],
            "vx": vel[:, 0],
            "vy": vel[:, 1],
            "time": t
        })

        path = os.path.join(output_dir, f"{name}.csv")
        df.to_csv(path, index=False)
        print(f"Saved: {path}")
        
save_all_datasets(dataset)